**mettre data sur le disque local pour chargement plus rapide**






In [1]:
from google.colab import drive
drive.mount('/content/drive')
base_path = "drive/MyDrive/MVA/S2/DLMI/kaggle_comp/"

src_train = base_path + 'train.h5'
src_val = base_path + 'val.h5'
src_test = base_path + 'test.h5'


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**télécharge+ set clearml**

In [2]:
#Install dependencies and set clearml
!pip install torchmetrics
!pip install clearml
%env CLEARML_WEB_HOST=https://app.clear.ml/
%env CLEARML_API_HOST=https://api.clear.ml/
%env CLEARML_FILES_HOST=https://files.clear.ml/
%env CLEARML_API_ACCESS_KEY=NY9YRCQ48P9CQO980VHZGKVYAB17A8
%env CLEARML_API_SECRET_KEY=rda3so8in2zuDRMARFZP9i4pDoOmxN2Os6sqKQKfWKUwC91YXbQKS0-W2RikIEj3a9E


env: CLEARML_WEB_HOST=https://app.clear.ml/
env: CLEARML_API_HOST=https://api.clear.ml/
env: CLEARML_FILES_HOST=https://files.clear.ml/
env: CLEARML_API_ACCESS_KEY=NY9YRCQ48P9CQO980VHZGKVYAB17A8
env: CLEARML_API_SECRET_KEY=rda3so8in2zuDRMARFZP9i4pDoOmxN2Os6sqKQKfWKUwC91YXbQKS0-W2RikIEj3a9E


**import dependencies and utils**

In [3]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch.optim.lr_scheduler import CyclicLR
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import models
from torchvision.transforms import ToTensor

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import KFold

import h5py
from tqdm.notebook import tqdm

**load all images**

In [4]:
def load_all_images(list_paths):

    all_images = []
    all_labels = []
    all_centers = []

    for path in list_paths:
        print("Loading dataset:", path)
        with h5py.File(path, 'r') as hdf:
            keys = list(hdf.keys())
            for key in tqdm(keys):
                img_array = np.array(hdf[key]['img'])  # shape (3, 96, 96)
                label = int(np.array(hdf[key]['label']))
                center = int(np.array(hdf[key]['metadata'])[0])

                all_images.append(img_array)
                all_labels.append(label)
                all_centers.append(center)

    # Conversion en tenseurs PyTorch
    images_np = np.stack(all_images, axis=0)   # shape (N, 3, 96, 96)
    labels_np = np.array(all_labels, dtype=np.int64)
    centers_np = np.array(all_centers, dtype=np.int64)

    images_t = torch.from_numpy(images_np)
    labels_t = torch.from_numpy(labels_np)
    centers_t = torch.from_numpy(centers_np)

    return images_t, labels_t, centers_t


**check**

In [5]:
def print_label_distribution_per_center(centers, labels):
    import numpy as np
    centers = np.array(centers)
    labels = np.array(labels)
    unique_centers = np.unique(centers)
    for center_id in unique_centers:
        mask = (centers == center_id)
        labels_for_center = labels[mask]
        unique_labels, counts = np.unique(labels_for_center, return_counts=True)
        distribution_str = ", ".join(f"Label {l}: {c}" for l, c in zip(unique_labels, counts))
        print(f"Center {center_id} -> {distribution_str}")


In [6]:
import albumentations as A
import torch
import numpy as np

def aug_train(p=1):
    return A.Compose([
        A.Resize(224, 224),
        A.HorizontalFlip(),
        A.VerticalFlip(),
        A.RandomRotate90(),
        A.Transpose(),
        A.Affine(
            scale=(0.5, 1.5),
            translate_percent=(0.0, 0.0625),
            rotate=(-45, 45),
            p=0.75
        ),
        A.OpticalDistortion(),
        A.GridDistortion(),
        A.RandomBrightnessContrast(p=0.3),
        A.RandomGamma(p=0.3),
        A.OneOf([
            A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=0.1, val_shift_limit=0.1, p=0.3),
            A.ChannelShuffle(p=0.3),
            A.CLAHE(p=0.3),
        ])
    ], p=p)

def aug_val(p=1):
    return A.Compose([
        A.Resize(224, 224)
    ], p=p)


In [7]:
def make_tta(image: np.ndarray):
    image_tta = np.zeros((4, *image.shape), dtype=image.dtype)

    # Original
    image_tta[0] = image

    # Horizontal Flip
    aug = A.HorizontalFlip(p=1)
    image_aug = aug(image=image)['image']
    image_tta[1] = image_aug

    # Vertical Flip
    aug = A.VerticalFlip(p=1)
    image_aug = aug(image=image)['image']
    image_tta[2] = image_aug

    # Transpose
    aug = A.Transpose(p=1)
    image_aug = aug(image=image)['image']
    image_tta[3] = image_aug

    return image_tta


In [8]:
class MyDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        """
        images : liste ou tensor de shape (N, 3, 96, 96)
        labels : liste ou tensor de shape (N,)
        transform : pipeline albumentations (A.Compose)
        """
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        """
        Retourne (image, label).
        image : Tensor shape (3, H, W) après l'albumentation.
        label : 0 ou 1
        """
        image = self.images[idx]    # shape (3, 96, 96) en Tensor
        label = self.labels[idx].item()  # int

        # Passage en np.array (H, W, C) pour Albumentations
        image_np = image.permute(1, 2, 0).numpy().astype(np.float32)  # (96, 96, 3)

        if self.transform:
            augmented = self.transform(image=image_np)
            image_torch = augmented["image"]
        else:
            image_torch = torch.from_numpy(image_np).permute(2, 0, 1)

        return image_torch, label

In [9]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


In [10]:
def validate_one_epoch(model, val_loader, device):
    """
    Évaluation "classique" (sans TTA)
    """
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    criterion = nn.BCEWithLogitsLoss()  # ex. binaire
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device, dtype=torch.float)            # (B, 3, 224, 224)
            labels = labels.to(device, dtype=torch.float).unsqueeze(1)  # (B, 1)

            outputs = model(images)              # (B, 1)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)

            preds = (torch.sigmoid(outputs) >= 0.5).float()          # (B, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_loss = running_loss / total
    val_acc = correct / total
    return val_loss, val_acc

In [12]:
"""def validate_one_epoch_tta(model, val_loader, device):

    #Évaluation AVEC TTA.
    #On transforme chaque image en 4 vues, puis on fait la moyenne des prédictions.

    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    val_aug = aug_val()
    criterion = nn.BCEWithLogitsLoss()

    with torch.no_grad():
        for images, labels in tqmd(val_loader):
            # images shape: (B, 3, 224, 224) en Tensor
            # On repasse temporairement en (B, 224, 224, 3) pour créer le TTA
            images_np = images.permute(0, 2, 3, 1).cpu().numpy()  # (B, H, W, C)

            bs = images_np.shape[0]
            labels = labels.to(device, dtype=torch.float).unsqueeze(1)  # (B, 1)
            preds_total = []

            for i in range(bs):
                # On récupère l'image i, shape (224, 224, 3)
                single_img = images_np[i]
                # On génère 4 vues
                tta_imgs = make_tta(single_img)  # (4, 224, 224, 3)

                # On applique la transform val_aug + ToTensorV2() sur chaque vue
                logits_views = []
                for view_i in range(tta_imgs.shape[0]):
                    augmented = val_aug(image=tta_imgs[view_i])       # dict
                    tensor_view = ToTensorV2()(image=augmented["image"])["image"]
                    # tensor_view shape (3, 224, 224)
                    tensor_view = tensor_view.unsqueeze(0).to(device, dtype=torch.float)  # (1, 3, 224, 224)
                    logit = model(tensor_view)  # (1, 1)
                    logits_views.append(logit)

                # Moyenne des 4 prédictions
                logits_views = torch.cat(logits_views, dim=0)  # (4, 1)
                avg_logit = logits_views.mean(dim=0, keepdim=True)  # (1, 1)
                preds_total.append(avg_logit)

            # On concatène toutes les prédictions du batch
            preds_total = torch.cat(preds_total, dim=0)  # (B, 1)
            loss = criterion(preds_total, labels)
            running_loss += loss.item() * bs

            # Calcul accuracy
            pred_bin = (torch.sigmoid(preds_total) >= 0.5).float()
            correct += (pred_bin == labels).sum().item()
            total += labels.size(0)

    val_loss = running_loss / total
    val_acc = correct / total
    return val_loss, val_acc"""

from tqdm import tqdm

def validate_one_epoch_tta(model, val_loader, device, nb_tta_views=4):
    """
    Évaluation AVEC TTA plus efficace :
     - On transforme chaque image en `nb_tta_views` vues (ex: 4).
     - On envoie toutes les vues d'un batch en une seule passe dans le réseau.
     - On fait la moyenne des prédictions pour chaque image du batch.
    """
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    val_aug = aug_val()
    criterion = nn.BCEWithLogitsLoss()

    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc="Validation with TTA"):
            # images.shape: (B, 3, 224, 224)
            bs = images.size(0)
            labels = labels.to(device, dtype=torch.float).unsqueeze(1)  # (B, 1)

            # On passe de (B, 3, H, W) à (B, H, W, 3) pour générer les TTA
            images_np = images.permute(0, 2, 3, 1).cpu().numpy()  # => (B, H, W, C)

            # ===== 1) On génère toutes les vues TTA pour ce batch, en listant tout =====
            all_tta_views = []
            for i in range(bs):
                # make_tta() => renvoie un np.array de shape (nb_tta_views, H, W, C)
                tta_imgs = make_tta(images_np[i])
                # On applique val_aug + ToTensorV2() sur chaque vue
                for t in range(nb_tta_views):
                    augmented = val_aug(image=tta_imgs[t])  # dict {"image": np.array, ...}
                    tensor_view = ToTensorV2()(image=augmented["image"])["image"]
                    # tensor_view shape : (3, H, W)
                    all_tta_views.append(tensor_view)

            # ===== 2) On empile toutes les vues pour former un batch unique =====
            # => shape (B*nb_tta_views, 3, H, W)
            tta_batch = torch.stack(all_tta_views, dim=0).to(device, dtype=torch.float)

            # ===== 3) On fait UNE seule passe dans le modèle =====
            logits = model(tta_batch)  # => shape (B*nb_tta_views, 1)

            # ===== 4) On regroupe par blocs de nb_tta_views pour chaque image initiale =====
            # => reshape en (B, nb_tta_views, 1)
            logits = logits.view(bs, nb_tta_views, -1)

            # On calcule la moyenne sur la dimension des vues => shape (B, 1)
            avg_logits = logits.mean(dim=1)

            # ===== 5) Calcul du loss / accuracy =====
            loss = criterion(avg_logits, labels)
            running_loss += loss.item() * bs

            pred_bin = (torch.sigmoid(avg_logits) >= 0.5).float()
            correct += (pred_bin == labels).sum().item()
            total += bs

    val_loss = running_loss / total
    val_acc = correct / total
    return val_loss, val_acc


In [16]:


def train_kfold(images, labels, n_splits=4, seed=42, num_epochs=5, batch_size=8, device='cuda'):
    set_seed(seed)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)

    # Initialisation ClearML (à adapter si déjà fait ailleurs)
    from clearml import Task
    task = Task.init(project_name = 'ensemble_learning', task_name = 'DLMI')
    logger = task.get_logger()  # On récupère le logger ClearML

    for fold, (train_idx, val_idx) in enumerate(kf.split(images)):
        print(f'===== FOLD {fold+1}/{n_splits} =====')

        # Sous-ensembles
        train_images = [images[i] for i in train_idx]
        train_labels = [labels[i] for i in train_idx]
        val_images   = [images[i] for i in val_idx]
        val_labels   = [labels[i] for i in val_idx]

        # Datasets
        train_dataset = MyDataset(
            train_images, train_labels,
            transform=A.Compose([
                aug_train(p=1),
                ToTensorV2()
            ])
        )

        val_dataset = MyDataset(
            val_images, val_labels,
            transform=A.Compose([
                aug_val(p=1),
                ToTensorV2()
            ])
        )
        print("done creating datasets")

        # DataLoaders
        train_loader = DataLoader(train_dataset, batch_size=batch_size,
                                  shuffle=True, num_workers=2, pin_memory=True)
        val_loader   = DataLoader(val_dataset,   batch_size=batch_size,
                                  shuffle=False, num_workers=2, pin_memory=True)
        print("done creating dataloaders")

        # Modèle : ResNet34 pré-entraîné
        model = models.resnet34(pretrained=True)
        num_ftrs = model.fc.in_features
        model.fc = nn.Linear(num_ftrs, 1)
        model = model.to(device)
        print("done loading pretrained model")

        # Optimiseur, Loss & Scheduler
        criterion = nn.BCEWithLogitsLoss()
        optimizer = optim.Adam(model.parameters(), lr=1e-3)
        steps_per_epoch = len(train_loader)
        scheduler = CyclicLR(
            optimizer, base_lr=1e-4, max_lr=1e-3,
            step_size_up=steps_per_epoch * 2,  # Nombre d'iters pour atteindre max_lr
            mode='triangular', cycle_momentum=False
        )

        best_val_loss = float('inf')
        best_epoch = 0

        for epoch in range(num_epochs):
            model.train()
            running_loss = 0.0
            correct = 0
            total = 0

            # Ajout de tqdm pour suivre la progression durant les batches
            for i, (images_batch, labels_batch) in enumerate(
                tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
            ):
                images_batch = images_batch.to(device, dtype=torch.float)
                labels_batch = labels_batch.to(device, dtype=torch.float).unsqueeze(1)

                optimizer.zero_grad()
                outputs = model(images_batch)
                loss = criterion(outputs, labels_batch)
                loss.backward()
                optimizer.step()
                scheduler.step()

                running_loss += loss.item() * images_batch.size(0)
                preds = (torch.sigmoid(outputs) >= 0.5).float()
                correct += (preds == labels_batch).sum().item()
                total += labels_batch.size(0)

            # Calcul des métriques sur le train
            train_loss = running_loss / total
            train_acc  = correct / total

            # Validation (TTA)
            val_loss, val_acc = validate_one_epoch_tta(model, val_loader, device)

            # ==> Log ClearML <==
            print(f"[Epoch {epoch+1}/{num_epochs}] "
                  f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
                  f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

            logger.report_scalar("train_loss", "train_loss", iteration=epoch, value=train_loss)
            logger.report_scalar("train_accuracy", "train_accuracy", iteration=epoch, value=train_acc)
            logger.report_scalar("val_loss", "val_loss", iteration=epoch, value=val_loss)
            logger.report_scalar("val_accuracy", "val_accuracy", iteration=epoch, value=val_acc)

            # Sauvegarde du meilleur modèle
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_epoch = epoch
                torch.save(model.state_dict(), base_path + f"best_model_fold{fold}.pth")

        print(f"Fin du fold {fold+1}. Meilleure val_loss: {best_val_loss:.4f} à l'époque {best_epoch+1}.\n")


In [14]:
images, labels, centers = load_all_images([src_train, src_val])

Loading dataset: drive/MyDrive/MVA/S2/DLMI/kaggle_comp/train.h5


100%|██████████| 100000/100000 [01:24<00:00, 1186.68it/s]


Loading dataset: drive/MyDrive/MVA/S2/DLMI/kaggle_comp/val.h5


100%|██████████| 34904/34904 [00:29<00:00, 1164.88it/s]


In [ ]:

device = "cuda" if torch.cuda.is_available() else "cpu"

# Lancez l'entraînement en K-Fold
train_kfold(
    images,
    labels,
    n_splits=4,   # par exemple 4 folds
    seed=42,
    num_epochs=10,
    batch_size=32,
    device=device
)

ClearML Task: created new task id=e0c73543e966460196b38e641ac649ce
2025-03-31 12:41:35,969 - clearml.Task - INFO - Storing jupyter notebook directly as code
ClearML results page: https://app.clear.ml/projects/3b7efa20b0f4475aae92df0edb2da66c/experiments/e0c73543e966460196b38e641ac649ce/output/log
===== FOLD 1/4 =====
done creating datasets
done creating dataloaders


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning:

The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning:

Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet34_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet34_Weights.DEFAULT` to get the most up-to-date weights.



2025-03-31 12:41:47,074 - clearml.model - INFO - Selected model id: ed95bfad0ec14715a8c23634042e05f4
done loading pretrained model


Epoch 1/10:  35%|███▌      | 1112/3162 [02:56<05:20,  6.40it/s]

ClearML Monitor: Could not detect iteration reporting, falling back to iterations as seconds-from-start


Validation with TTA: 100%|██████████| 1054/1054 [05:50<00:00,  3.01it/s]


[Epoch 1/10] Train Loss: 0.1862 | Train Acc: 0.9274 | Val Loss: 0.1395 | Val Acc: 0.9444


Epoch 2/10:   3%|▎         | 84/3162 [00:13<08:03,  6.36it/s]

# Eval part

In [ ]:
linear_probing.load_state_dict(torch.load('best_model.pth', weights_only=True))
linear_probing.eval()
linear_probing.to(device)
prediction_dict = {}

In [ ]:
with h5py.File(TEST_IMAGES_PATH, 'r') as hdf:
    test_ids = list(hdf.keys())

In [ ]:
solutions_data = {'ID': [], 'Pred': []}
with h5py.File(TEST_IMAGES_PATH, 'r') as hdf:
    for test_id in tqdm(test_ids):
        img = preprocessing(torch.tensor(np.array(hdf.get(test_id).get('img')))).unsqueeze(0).float()
        pred = linear_probing(feature_extractor(img.to(device))).detach().cpu()
        solutions_data['ID'].append(int(test_id))
        solutions_data['Pred'].append(int(pred.item() > 0.5))
solutions_data = pd.DataFrame(solutions_data).set_index('ID')
solutions_data.to_csv(base_path + 'baseline.csv')